In [0]:
from pyspark.sql import functions as f,Window
import re
from datetime import datetime
from pyspark.sql.types import *

c_schema = StructType([
    StructField('customer_id',IntegerType()),
    StructField('customer_name',StringType()),
    StructField('city',StringType()),
    StructField('customer_segment',StringType()),
    StructField('email',StringType()),
    StructField('phone',StringType())
])

files = dbutils.fs.ls("/Volumes/madstagescd2/madstage/madstage/incrementalLoad/")
customers_files = []
for file in files:
    match = re.search(r'(\d{4}-\d{2}-\d{2})',file.name)
    if file.name.startswith('customer'):
        if match:
            file_date = datetime.strptime(match.group(1),"%Y-%m-%d")
            customers_files.append((file_date,file.path))
if customers_files:
    latest_file = max(customers_files,key=lambda x:x[0])
    customer = (spark.read.format('csv').option('header','true').schema(c_schema).load(latest_file[1]))

customer = customer.dropna(subset=('customer_id'))
customer = customer.dropDuplicates()
customer = customer.fillna(
    {
    'customer_name':'Unknown',
    'city':'Unknown',
    'customer_segment':'Unknown',
    'email':'Unknown',
    'phone':'Unknown'
    }
)

phone_regex = r'[^0-9]'
customer = customer.withColumn('customer_name',f.initcap(f.trim(f.col('customer_name')))).withColumn('city',f.initcap(f.trim(f.col('city')))).withColumn('customer_segment',f.initcap(f.trim(f.col('customer_segment')))).withColumn('email',f.lower(f.trim(f.col('email')))).withColumn('phone',f.regexp_replace(f.col('phone'),phone_regex,''))

email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'
email_validation =(f.col('email').rlike(email_regex))
phone_validation = (f.length(f.col('phone'))==10)

customer = customer.withColumn('dataValidation',f.when((email_validation)&(phone_validation),'Pass').otherwise('Fail'))

customer_df = customer.filter(f.col('dataValidation')=='Pass').drop('dataValidation')

#scd2 implementaion

legacy_df = (spark.read.table('madstagescd2.madstage.silver_customers').filter(f.col('is_current')=='Y'))

changed_df = (legacy_df.alias('t').join(customer_df.alias('s'),on='customer_id',how='inner').filter(
    (~f.col('t.customer_name').eqNullSafe(f.col('s.customer_name'))) | (~f.col('t.city').eqNullSafe(f.col('s.city')))
    | (~f.col('t.customer_segment').eqNullSafe(f.col('s.customer_segment'))) | (~f.col('t.email').eqNullSafe(f.col('s.email'))) | (~f.col('t.phone').eqNullSafe(f.col('s.phone')))
).select('s.*'))

changed_df.createOrReplaceTempView('changed_df')

new_df = (customer_df.alias('c').join(legacy_df.alias('l'),on='customer_id',how='left_anti'))
                                                                                            
spark.sql("""
          merge into madstagescd2.madstage.silver_customers as t
          using changed_df as s
          on t.customer_id = s.customer_id
          and t.is_current = 'Y'
          when matched then
          update set
          t.is_current = 'N',
          t.updated_dt = CAST(current_timestamp() as string) 
          """)

new_version_df = (new_df.unionByName(changed_df).withColumn('is_current',f.lit('Y')).withColumn('updated_dt',f.lit(None).cast('string')).withColumn('created_dt',f.current_timestamp().cast('string')))

new_version_df.createOrReplaceTempView('new_version_df')

spark.sql("""
          merge into madstagescd2.madstage.silver_customers as t
          using new_version_df as s
          on t.customer_id = s.customer_id
          and t.is_current = 'Y'
          when not matched then
          insert 
          (customer_id,customer_name,city,customer_segment,email,phone,is_current,updated_dt,created_dt)
          values(s.customer_id,s.customer_name,s.city,s.customer_segment,s.email,s.phone,s.is_current,s.updated_dt,s.created_dt)
          """)

print('SCD2 Implemented on customer table')
